# 2단계 순화·답변 LLM — Qwen2.5-7B QLoRA (Colab, Unsloth)

합성데이터(`synth.jsonl`)의 **원문→순화** + **원문→답변** 쌍으로 Qwen2.5-7B를 파인튜닝.
두 작업은 **user 메시지 지시문**으로 명확히 구분(순화/답변). GGUF로 export → `ollama create`.

- 런타임: **GPU (T4 이상)**. 7B QLoRA 4bit 는 T4에서 학습 가능.


## 1) Unsloth 설치


In [ ]:
!pip -q install unsloth

## 2) 합성데이터 업로드
`data/processed/synth.jsonl`(reply 포함) 업로드. 3번 셀이 '2000'이면 정상.


In [ ]:
from google.colab import files
up = files.upload()
DATA = [k for k in up if k.endswith('.jsonl')][0]
print('데이터:', DATA)

## 3) 학습 예시 구성 (순화/답변 작업 구분 지시문 포함)


In [ ]:
import json

REWRITE_SYS = ('반드시 한국어로만 작성한다. 너는 학부모 민원을 공적인 표현으로 다듬는다. '
  '학부모가 제기한 문제와 요구를 감정·공격 표현을 빼고 중립적인 한두 문장으로 요약하라. '
  '교사의 답변·약속을 쓰지 말고 학부모가 무엇을 문제삼고 무엇을 원하는지만 적는다. 요약문만 출력하라.')
DRAFT_SYS = ('반드시 한국어로만 작성한다. 너는 교사가 학부모에게 보낼 답변 초안을 쓴다. '
  '학부모의 감정과 자녀에게 먼저 공감하고, 사실 확인 전이므로 잘못을 인정하지 말고 확인하겠다는 태도로, '
  '구체적 후속(면담·확인·연락)을 제안하라. 교권·법령 같은 학교 방어 논리를 언급하지 말고, '
  '4~5문장으로 따뜻하고 간결하게 본문만 출력하라.')

# 작업 구분 지시문 — 백엔드(app/nlp/rewrite.py)와 '정확히 일치'해야 함
SUNHWA_INSTR = '다음 학부모 민원을 공적인 표현으로 요약해줘.\n민원: '
DAPBYEON_INSTR = '다음 학부모 민원에 대해 교사가 학부모에게 보낼 답변 초안을 작성해줘.\n민원: '

examples = []
for line in open(DATA, encoding='utf-8'):
    r = json.loads(line)
    examples.append({'messages':[{'role':'system','content':REWRITE_SYS},
                                 {'role':'user','content':SUNHWA_INSTR + r['original']},
                                 {'role':'assistant','content':r['rewritten']}]})
    if r.get('reply'):
        examples.append({'messages':[{'role':'system','content':DRAFT_SYS},
                                     {'role':'user','content':DAPBYEON_INSTR + r['original']},
                                     {'role':'assistant','content':r['reply']}]})
print('학습 예시:', len(examples), '개 (순화+답변) — 2000 이어야 정상')

## 4) 모델 로드 (Qwen2.5-7B 4bit + LoRA)


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length=1024, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

## 5) chat 템플릿 적용


In [ ]:
from datasets import Dataset
ds = Dataset.from_list(examples)
def fmt(b):
    return {'text':[tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                    for m in b['messages']]}
ds = ds.map(fmt, batched=True, remove_columns=ds.column_names)
print(ds[0]['text'][:300])

## 6) 학습


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds, dataset_text_field='text',
    max_seq_length=1024,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=20, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=42, output_dir='out', report_to=[]))
trainer.train()

## 7) 빠른 테스트 — 순화/답변이 서로 다르게 나와야 정상


In [ ]:
FastLanguageModel.for_inference(model)
def gen(system, instr, user):
    msgs=[{'role':'system','content':system},{'role':'user','content':instr+user}]
    ids=tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out=model.generate(input_ids=ids, max_new_tokens=200, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

민원='우리 애 발표시켜서 자존감 떨어뜨렸는데 책임지실건가요?'
print('순화:', gen(REWRITE_SYS, SUNHWA_INSTR, 민원))
print()
print('답변:', gen(DRAFT_SYS, DAPBYEON_INSTR, 민원))

## 8) GGUF export → 다운로드


In [ ]:
model.save_pretrained_gguf('aitg_sunhwa', tokenizer, quantization_method='q4_k_m')

import glob, os
from google.colab import files
gguf = glob.glob('aitg_sunhwa_gguf/*.gguf')[0]
gguf_name = os.path.basename(gguf)
print('GGUF:', gguf, round(os.path.getsize(gguf)/1e9,2), 'GB')

modelfile = f'''FROM ./{gguf_name}
TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
{{{{ .Response }}}}<|im_end|>"""
PARAMETER temperature 0.3
PARAMETER stop "<|im_end|>"
'''
open('aitg_sunhwa_gguf/Modelfile','w').write(modelfile)
!cd aitg_sunhwa_gguf && zip -q ../aitg_sunhwa_ollama.zip {gguf_name} Modelfile
files.download('aitg_sunhwa_ollama.zip')
print('완료')

## 9) 맥에서 Ollama 등록
```bash
cd ~/Downloads && unzip -o aitg_sunhwa_ollama.zip -d aitg_sunhwa && cd aitg_sunhwa
ollama create aitg-sunhwa -f Modelfile
OLLAMA_MODEL=aitg-sunhwa CLASSIFIER_MODEL_DIR=~/Downloads/klue-roberta-clf uvicorn app.main:app --reload
```
